# 00 · NCBI E-utilities  -  the deep tutorial

**Goal.** Give you enough context on NCBI, its databases, its API, and the
biomedical-domain vocabulary that you can pull *any* PubMed slice you want
and know exactly what the numbers mean.

**What this notebook covers**

1. What NCBI actually is (the institution)
2. The NCBI database landscape  -  which of the 40+ databases matter for which questions
3. The Entrez search model (query syntax, field tags, MeSH)
4. The nine E-utilities and when to use each
5. Rate limits, API keys, and being a polite API citizen
6. The WebEnv / history server pattern for large queries
7. Parsing PubMed XML into tidy DataFrames
8. A biology glossary  -  PMID, PMCID, DOI, MeSH, ORCID, GenBank, etc.
9. A worked example: publications on CRISPR by year and country
10. Reproducibility notes: what to cite, what to record

If you already know some of this, skip ahead  -  each section is standalone.


## 1. What is NCBI?

**NCBI**  -  the National Center for Biotechnology Information  -  is a division
of the U.S. National Library of Medicine (NLM), which is itself part of the
National Institutes of Health (NIH). Founded in 1988, it exists to
"create automated systems for storing and analyzing knowledge about molecular
biology, biochemistry, and genetics."

That mission is why NCBI hosts a sprawling family of public databases and
tools: PubMed for literature, GenBank for DNA sequences, BLAST for
sequence similarity search, dbSNP for genetic variation, and dozens more.
When people say "NCBI" in a data-analysis context, they almost always mean
one or more of those databases and their APIs.

**Why this matters for research design.** NCBI data is authoritative,
public, and free  -  but also *messy* in specific ways (unstructured
affiliation strings, inconsistent MeSH indexing lag, deprecated identifiers).
Understanding the institution's history helps you understand its data
peculiarities. For example: MeSH indexing is done by humans at NLM, which
is why very recent papers often show up without MeSH terms.

**Docs to bookmark:**
- NCBI home: <https://www.ncbi.nlm.nih.gov/>
- Entrez book (canonical reference): <https://www.ncbi.nlm.nih.gov/books/NBK3837/>
- E-utilities book: <https://www.ncbi.nlm.nih.gov/books/NBK25497/>


## 2. The database landscape  -  a quick map

NCBI hosts ~40 databases behind Entrez. For literature and bibliometric
research these five are the most relevant:

| Database   | What's in it                           | Typical use for panel analysis |
|-----------|----------------------------------------|--------------------------------|
| **PubMed**    | Citations + abstracts (>35M records)   | Publication counts, MeSH trends, retractions |
| **PMC**       | Full-text open-access articles         | Full-text mining, section-level analysis |
| **Gene**      | One record per gene per organism       | Which genes/organisms get studied over time |
| **ClinVar**   | Variant + clinical significance        | Growth of variant knowledge; equity in reporting |
| **ClinicalTrials.gov** (via linkage) | Trials + results | Linking trials to publications, results-reporting gaps |

For sequence-heavy research, add these:

| Database   | What's in it                           |
|-----------|----------------------------------------|
| **GenBank / nuccore** | DNA sequences |
| **Protein**   | Protein sequences |
| **SRA**       | Raw sequencing reads |
| **BioProject / BioSample** | Study and sample metadata |

The full list is at <https://www.ncbi.nlm.nih.gov/guide/all/>  - 
it's worth a scroll even if you'll only use two of them.


## 3. Entrez query syntax  -  how PubMed search actually works

The search box on PubMed and every ESearch call use the same language.
Understanding it means you spend hours less debugging.

**Field tags**  -  every query term can be scoped to a field:

| Tag        | Field                                        |
|-----------|----------------------------------------------|
| `[tiab]`   | Title + Abstract                             |
| `[mesh]` or `[mh]` | MeSH heading                          |
| `[au]`     | Author                                       |
| `[ta]`     | Journal (title abbreviation)                 |
| `[dp]`     | Publication date                             |
| `[pt]`     | Publication type ("Retracted Publication", "Review", "Clinical Trial") |
| `[la]`     | Language                                     |
| `[ad]`     | Affiliation                                  |
| `[la]`     | Language                                     |

**Booleans:** `AND`, `OR`, `NOT` (must be UPPERCASE). Parentheses group.

**Example:**
```
("CRISPR"[tiab] OR "CRISPR-Cas9"[mesh]) AND "2020"[dp]:"2024"[dp] AND English[la]
```

**MeSH terms** are the controlled vocabulary NLM uses to index articles.
They're much more stable than free-text keywords  -  "myocardial infarction"
finds the same papers whether the author wrote "heart attack" or
"MI" or "STEMI". Browse the MeSH tree at <https://meshb.nlm.nih.gov/>.

**A key subtlety:** without a field tag, Entrez runs **Automatic Term
Mapping (ATM)**  -  it tries to be helpful by mapping your string to MeSH
terms, author names, and phrase indices in a specific order. Great for
casual search, treacherous for research: you may get different results
than you expect. When in doubt, add explicit tags.


## 4. The nine E-utilities  -  and which four you'll actually use

E-utilities are HTTP endpoints under `https://eutils.ncbi.nlm.nih.gov/entrez/eutils/`.
Each returns XML by default, JSON when `retmode=json` is available.

| Utility     | What it does                                       | Use it for |
|-------------|----------------------------------------------------|------------|
| **ESearch** | Query text → list of UIDs                          | The starting point of nearly every workflow |
| **ESummary**| UID → structured summary (short metadata)          | Fast listings, spreadsheets |
| **EFetch**  | UID → full record (XML or text)                    | Anything that needs abstracts, MeSH, affiliations |
| **EPost**   | Upload a list of UIDs to the server as a query     | When you already have UIDs from elsewhere |
| **ELink**   | UIDs in one DB → related UIDs in another           | "Papers linked to this ClinicalTrials.gov entry" |
| **EInfo**   | Metadata about a database (fields, links)          | Discovery |
| **EGQuery** | Query across all databases at once                 | Rarely used in analysis |
| **ESpell**  | Spelling suggestions                                | Interactive UIs |
| **ECitMatch** | Match citations to PMIDs                          | Cleaning reference lists |

For panel analysis you'll live in **ESearch → EFetch** with WebEnv history
in between. `src/ncbi_client.py` wraps exactly that.


## 5. Rate limits and API keys  -  the polite-citizen rules

NCBI publishes hard limits:

- **Without an API key:** 3 requests/second per IP.
- **With a free API key:** 10 requests/second per IP.
- No overall daily quota, but heavy jobs should run outside 9am-5pm US Eastern (their peak human-user window).

**Get a key in 3 minutes:**
1. Sign in at <https://www.ncbi.nlm.nih.gov/account/>.
2. Go to Account Settings → API Key Management.
3. Click "Create an API Key". It's a 40-character hex string.
4. Never commit it to a public repo. This project's `.gitignore` ignores `.env`.

Set it in your shell before starting Jupyter:

```bash
export NCBI_API_KEY="paste-your-key-here"
export NCBI_EMAIL="you@example.com"   # NCBI recommends including a contact email
```

Our client reads both from the environment. If you skip both, it still
works  -  you just get 3 req/s and NCBI can't reach you if your queries
misbehave.


In [ ]:
import os, sys
sys.path.insert(0, "..")

from src.ncbi_client import API_KEY, EMAIL, REQ_INTERVAL
print(f"API key detected: {bool(API_KEY)}")
print(f"Email detected:   {bool(EMAIL)}")
print(f"Throttle: 1 request every {REQ_INTERVAL:.3f}s  ({1/REQ_INTERVAL:.1f} req/s)")


## 6. First real call  -  one ESearch, no library magic

Now send an ESearch by hand so the mechanics are transparent. We'll
ask: "how many PubMed papers mention CRISPR in title or abstract,
published in 2023?"


In [ ]:
import requests
url = ("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
       "?db=pubmed"
       "&term=CRISPR%5Btiab%5D+AND+2023%5Bdp%5D"
       "&retmode=json&retmax=0")
r = requests.get(url, timeout=30).json()
print("count :", r["esearchresult"]["count"])
print("keys  :", list(r["esearchresult"])[:10])


### `esearchresult` keys, decoded

- `count`  -  total hits.
- `idlist`  -  up to `retmax` PMIDs. If you set `retmax=0`, you only get the count (fast).
- `webenv` and `querykey`  -  the server-side handles. See section 7.
- `translationset`, `querytranslation`  -  how ATM interpreted your query.
- `errorlist`, `warninglist`  -  worth checking on real queries.


## 7. The WebEnv / history-server pattern  -  for pulls >10K records

`retmax` caps at 10,000 per ESearch. For anything bigger, use the history
server:

1. First ESearch with `usehistory=y`. NCBI stores the full ID list on
   its side and returns `webenv` + `querykey`.
2. Loop EFetch (or ESummary) calls with those handles and `retstart`
   stepping through in batches of, say, 200.

Our client hides this in `pubmed_query_to_dataframe()`. Here's what
step 1 looks like at the raw level:


In [ ]:
from src.ncbi_client import esearch, efetch_pubmed_xml, articles_to_dataframe

search = esearch(
    'CRISPR[tiab] AND 2023[dp]',
    mindate="2023/01/01", maxdate="2023/12/31",
    retmax=0, use_history=True,
)
print(f"Hits: {search['count']:,}")
print(f"webenv:   {search['webenv'][:30]}...")
print(f"query_key: {search['query_key']}")


In [ ]:
# Step 2: EFetch 300 of them via the history handle (small pull for illustration).
xml_root = efetch_pubmed_xml(
    webenv=search["webenv"],
    query_key=search["query_key"],
    count=300,
    batch_size=200,
)
df = articles_to_dataframe(xml_root)
print("shape:", df.shape)
df[["pmid", "year", "journal", "n_authors", "country", "n_mesh"]].head()


## 8. Biology glossary  -  the acronyms you keep seeing

If you didn't grow up in biomedical labs, these come up constantly.

| Acronym    | What it is |
|-----------|------------|
| **PMID**   | PubMed identifier  -  a plain integer, unique per citation |
| **PMCID**  | PubMed Central identifier (starts with `PMC...`)  -  a full-text open-access article, when available |
| **DOI**    | Digital Object Identifier  -  publisher-side ID, universal across databases |
| **MeSH**   | Medical Subject Headings  -  NLM's controlled vocabulary |
| **NLM**    | National Library of Medicine  -  NCBI's parent |
| **NIH**    | National Institutes of Health  -  NLM's parent |
| **ORCID**  | Open Researcher and Contributor ID  -  a per-person unique ID |
| **GenBank**| NCBI's DNA sequence database |
| **RefSeq** | NCBI's curated ("reference") sequences |
| **BLAST**  | Basic Local Alignment Search Tool  -  sequence similarity search |
| **SRA**    | Sequence Read Archive  -  raw high-throughput sequencing reads |
| **dbSNP**  | Database of Single Nucleotide Polymorphisms |
| **ClinVar**| Clinical variant significance database |
| **CRISPR** | Clustered Regularly Interspaced Short Palindromic Repeats  -  a gene-editing technique/system |
| **RCT**    | Randomized Controlled Trial |
| **PubType**| PubMed's `PublicationType`  -  includes "Review", "Clinical Trial", "Retracted Publication" |

**Retraction vocabulary** (relevant for RQ3):
- A **Retracted Publication** is a paper that has been retracted.
- A **Retraction of Publication** is the *notice* that retracts it  -  a separate PMID that references the first.
  When you count retractions, use `Retracted Publication[pt]` (the paper itself), not the notice.


## 9. Worked example  -  publications on CRISPR by year and country

We'll pull every article tagged with the MeSH term "CRISPR-Cas Systems"
from 2013 (the year the term was added) to 2023, then aggregate by
(country, year). The country tag comes from our heuristic parser on
first-author affiliation strings.

**A caveat before you cite this.** MEDLINE affiliation data are messy:
older records often have only the first-author affiliation; some records
have no affiliation at all; multi-country collaborations are collapsed
to one country by our heuristic. Aggregate counts are directionally
correct but exact per-year rankings should be sanity-checked.


In [ ]:
from src.ncbi_client import pubmed_query_to_dataframe

# Small-ish pull to keep the notebook fast; max_records=5000 gives us a
# representative sample. For the RQ notebooks we'll pull the full set.
crispr = pubmed_query_to_dataframe(
    '"CRISPR-Cas Systems"[mesh] OR "CRISPR"[tiab]',
    mindate="2013/01/01", maxdate="2023/12/31",
    max_records=5000,
)
print(crispr.shape)
crispr.head()


In [ ]:
# Publications per year  -  the classic bibliometric curve.
per_year = crispr.dropna(subset=["year"]).groupby("year").size()
per_year.plot(kind="bar", figsize=(9, 4), title="CRISPR PubMed publications per year (sample)")


In [ ]:
# Top 10 countries in the sample.
crispr[crispr["country"].notna()]["country"].value_counts().head(10)


## 10. Reproducibility notes  -  what to record

For any bibliometric result you'll defend, log **all** of these:

1. **The exact query string**, including field tags and Boolean operators.
2. **The date range** and whether you filtered by `dp` (publication date), `edat` (Entrez date), or `pdat` (print date). These differ by weeks or months for the same paper.
3. **The date you ran the query.** PubMed grows and revises daily; a query re-run three months later will return more results.
4. **Whether you used your API key**, so the throttle behavior is reproducible.
5. **Which fields you extracted from the XML**  -  different parsers may extract different subsets.

Two citations to include when publishing:
- **NCBI itself:** NCBI Resource Coordinators. *Database resources of the National Center for Biotechnology Information.* Nucleic Acids Res. 2018;46:D8-D13.
- **E-utilities:** Sayers E. *The E-utilities In-Depth: Parameters, Syntax and More.* NCBI Books. 2010-.


## Ready for the RQ notebooks

You're set. Open `01_rq1_country_publication_panel.ipynb` next.
It uses everything above to build a country × year panel of biomedical
publications in a chosen field and fits a two-way fixed-effects regression.
